# barrierQP

This notebook solves a small dense, strictly convex quadratic program with the
Mehrotra predictor-corrector interior-point method in `barrierqp`. Its one idea is
factorization reuse: each iteration factors a single KKT matrix and reuses that
factorization for both the affine predictor and the centered corrector direction.

In [1]:
import numpy as np
import jax
jax.config.update("jax_enable_x64", True)   # dense KKT solves want float64
import barrierqp

In [2]:
# A pentagon: the square [-1, 1]^2 cut by the diagonal 0.625 (x1 + x2) <= 1.
G = np.array([[1., 0.], [0., 1.], [0.625, 0.625], [-1., 0.], [0., -1.]])
h = np.ones(5)
P = np.array([[2.0, 0.5], [0.5, 1.5]])
q = np.array([-2.7, -2.175])
A, b = np.zeros((0, 2)), np.zeros(0)          # no equality rows

print(f"variables:    {P.shape[0]}")
print(f"equalities:   {A.shape[0]}")
print(f"inequalities: {G.shape[0]}")

variables:    2
equalities:   0
inequalities: 5


In [3]:
result = barrierqp.Solver(P, q, A, b, G, h).solve()
x = np.asarray(result.x); z = np.asarray(result.z)
objective = float(0.5 * x @ P @ x + q @ x)
primal = float(np.max(np.maximum(G @ x - h, 0.0), initial=0.0))
kkt = float(np.max(np.abs(P @ x + q + G.T @ z), initial=0.0))

print(f"status:          {result.status}")
print(f"x:               [{x[0]:.6f}, {x[1]:.6f}]")
print(f"objective:       {objective:.6f}")
print(f"iterations:      {result.iterations}")
print(f"factorizations:  {result.factorizations}")
print(f"Newton solves:   {result.newton_solves}")
print(f"primal residual: {primal:.2e}")
print(f"KKT residual:    {kkt:.2e}")

status:          solved
x:               [0.850000, 0.750000]
objective:       -2.463125
iterations:      6
factorizations:  6
Newton solves:   12
primal residual: 0.00e+00
KKT residual:    8.47e-10


The optimizer lies on the diagonal edge of the feasible pentagon. The six
iterations correspond to six KKT factorizations and twelve direction solves --
affine and corrector share one factor per iteration -- so factorization reuse is
visible directly in the counters, without exposing any internal trace machinery.

In [4]:
# Change only the linear term q and solve again.
q2 = np.array([-1.0, 2.5])
r2 = barrierqp.Solver(P, q2, A, b, G, h).solve()
x2 = np.asarray(r2.x)

print(f"original x:         [{x[0]:.6f}, {x[1]:.6f}]")
print(f"modified x:         [{x2[0]:.6f}, {x2[1]:.6f}]")
print(f"original objective: {objective:.6f}")
print(f"modified objective: {float(0.5 * x2 @ P @ x2 + q2 @ x2):.6f}")
print(f"second status:      {r2.status}")
print(f"second iterations:  {r2.iterations}")

original x:         [0.850000, 0.750000]
modified x:         [0.750000, -1.000000]
original objective: -2.463125
modified objective: -2.312500
second status:      solved
second iterations:  6


The example is intentionally small. For an optional local timing experiment
against other solvers, see `bench.py`.